In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser

/tmp/ipykernel_2525/3423817975.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [2]:
# Load the textfile
loader = TextLoader("langchain_sample.txt")
raw_docs = loader.load()

# Split document into chunks
splitter = RecursiveCharacterTextSplitter(
    separators=["."],
    chunk_size = 500,chunk_overlap = 50
)
docs = splitter.split_documents(raw_docs)
docs

[Document(metadata={'source': 'langchain_sample.txt'}, page_content='LangChain is an open-source framework designed to simplify the construction of applications powered by\nlarge language models (LLMs), serving as the easiest starting point for developers while remaining \nflexible and production-ready. Its core philosophy rests on the belief that LLMs become far more powerful\nwhen combined with external data sources, and that future applications will become increasingly agentic—capable\nof reasoning and acting autonomously'),
 Document(metadata={'source': 'langchain_sample.txt'}, page_content='. LangChain addresses two central challenges: standardizing model \ninputs and outputs across different providers so developers can easily switch to the latest state-of-the-art\nmodels without vendor lock-in, and making it simple to use models for orchestrating complex flows that interact\nwith other data and computation through dynamically defined tools.The framework provides four main buildin

In [3]:
# FAISS and HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorstores = FAISS.from_documents(docs,embedding_model)

/tmp/ipykernel_2525/600654278.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [4]:
retriever = vectorstores.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k":3}
)

In [5]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from langchain_groq import ChatGroq

llm = ChatGroq(
     model="openai/gpt-oss-20b",
     temperature=0.7,
     api_key=os.getenv("GROQ_API_KEY")
 )
prompt = PromptTemplate.from_template("""
    Answer the question based on the context below.
    
    context:{context}
    Question:{input}"""
)

In [10]:
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

In [11]:
document_chain = create_stuff_documents_chain(llm,prompt)
rag_chain = create_retrieval_chain(retriever,document_chain)
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7ff75abcb320>, search_type='mmr', search_kwargs={'k': 3}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | PromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, template='\n    Answer the question based on the context below.\n\n    context:{context}\n    Question:{input}')
            | ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.1', 'langchain': '1.3.18'}}, profile={'name': 'GPT OSS 20B', 'release

In [12]:
query = {"input":"How can i build app using LLMS?"}
response = rag_chain.invoke(query)
print("Answer:\n", response["answer"][:500])

print("\nSource Documents:")
for i,doc in enumerate(response["context"]):
    print(f"\nDoc {i+1}: {doc.page_content[:150]}")

Answer:
 **Building an LLM‑powered app with LangChain – a quick‑start guide**

Below is a practical, step‑by‑step recipe that follows the core ideas in the context you provided.  
Feel free to skip or reorder steps to fit your project’s size and scope.

| Step | What to do | Why it matters | Typical tools / examples |
|------|------------|----------------|--------------------------|
| **1. Define the problem & data** | • Clarify the business goal (e.g., Q&A, summarisation, recommendation). <br>• Identify

Source Documents:

Doc 1: LangChain is an open-source framework designed to simplify the construction of applications powered by
large language models (LLMs), serving as the ea

Doc 2: . The choice of embedding model involves trade-offs between quality 
(often measured by the MTEB leaderboard), cost, latency, and dimensionality, with

Doc 3: .The framework provides four main building
blocks. Chains connect multiple operations in sequence, where the output of one step becomes the input